In [ ]:
!pip install "datasets<3.0.0" soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from scipy.signal import resample

from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav

# ── Config ───────────────────────────────────────────────────────────────────
NUM_SAMPLES = 15
MODEL_NAME  = "facebook/mms-tts-urd-script_arabic"

# ── Load TTS model + speaker encoder ─────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = VitsModel.from_pretrained(MODEL_NAME)
model.eval()
encoder   = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key):
    """Generate TTS, save MUSHRA audio triplets, compute Resemblyzer similarities."""
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i, sample in enumerate(samples_list[:NUM_SAMPLES]):
        try:
            text      = sample[text_key]
            ref_array = np.array(sample["audio"]["array"]).astype(np.float32)
            ref_sr    = sample["audio"]["sampling_rate"]
            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # 1. Reference
            sf.write(ref_path, ref_array, ref_sr)

            # 2. TTS generation
            inputs = tokenizer(text, return_tensors="pt")
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr     = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # 3. Anchor (8 kHz resample cycle)
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled   = resample(ref_array, target_len_8k)
            upsampled     = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # 4. Resemblyzer embeddings
            ref_wav = preprocess_wav(ref_path)
            gen_wav = preprocess_wav(gen_path)
            ref_emb = encoder.embed_utterance(ref_wav)
            gen_emb = encoder.embed_utterance(gen_wav)
            ref_self_emb = encoder.embed_utterance(ref_wav)   # same file → self-sim

            ref_gen_sim  = float(np.dot(ref_emb, gen_emb))
            ref_self_sim = float(np.dot(ref_emb, ref_self_emb))

            results.append({
                "sample_id":           i,
                "urdu_text":           text,
                "audio_duration_sec":  round(len(ref_array) / ref_sr, 2),
                "ref_gen_similarity":  round(ref_gen_sim,  4),
                "ref_self_similarity": round(ref_self_sim, 4),
            })
            print(f"  [{i+1:02d}/{NUM_SAMPLES}] ref_gen_sim={ref_gen_sim:.4f}  text={text[:40]}")
        except Exception as e:
            print(f"  ⚠️  Sample {i} failed: {e}")

    print(f"\n✅ {dataset_name}: {len(results)}/{NUM_SAMPLES} samples evaluated")
    return results


In [ ]:
from google.colab import userdata
try:
    hf_token = userdata.get("HF_TOKEN")
    print("✅ HF token loaded from Colab secrets")
except Exception:
    hf_token = None
    print("⚠️  HF_TOKEN not found in secrets — proceeding without token (may fail for gated datasets)")


In [ ]:
from datasets import load_dataset

# ── 3. Literary / Storytelling style (style == "BOOK") ───────────────────────
# data_dir="Urdu" keeps only Urdu shards; Kashmiri shards are excluded at source.
print("📖 Loading Humairawan/UrduSpeech  [style=BOOK] ...")

ds_book_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

book_samples = []
for sample in ds_book_stream:
    if str(sample.get("style", "")).strip().upper() == "BOOK":
        book_samples.append(sample)
    if len(book_samples) >= NUM_SAMPLES:
        break

print(f"  Collected {len(book_samples)} BOOK samples")
results_book = run_evaluation("humair_book", book_samples, text_key="text")


In [ ]:
# ── 4. Formal / Read speech (style == "WIKI") ────────────────────────────────
# Same dataset, different style filter — encyclopedic / formal narration.
print("📚 Loading Humairawan/UrduSpeech  [style=WIKI] ...")

ds_wiki_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

wiki_samples = []
for sample in ds_wiki_stream:
    if str(sample.get("style", "")).strip().upper() == "WIKI":
        wiki_samples.append(sample)
    if len(wiki_samples) >= NUM_SAMPLES:
        break

print(f"  Collected {len(wiki_samples)} WIKI samples")
results_wiki = run_evaluation("humair_wiki", wiki_samples, text_key="text")


In [ ]:
# ── 5. Conversational style (style == "CONV") ────────────────────────────────
# Same dataset, conversational sentences filter.
print("💬 Loading Humairawan/UrduSpeech  [style=CONV] ...")

ds_conv_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

conv_samples = []
for sample in ds_conv_stream:
    if str(sample.get("style", "")).strip().upper() == "CONV":
        conv_samples.append(sample)
    if len(conv_samples) >= NUM_SAMPLES:
        break

print(f"  Collected {len(conv_samples)} CONV samples")
results_conv = run_evaluation("humair_conv", conv_samples, text_key="text")


In [ ]:
import zipfile

# ── Combine results ───────────────────────────────────────────────────────────
all_data = []
for name, res in [
    ("humair_book", results_book),
    ("humair_wiki", results_wiki),
    ("humair_conv", results_conv),
]:
    for entry in res:
        entry["dataset"] = name
        all_data.append(entry)

df = pd.DataFrame(all_data)
df = df[["dataset", "sample_id", "urdu_text", "audio_duration_sec",
         "ref_gen_similarity", "ref_self_similarity"]]

# ── Summary table ─────────────────────────────────────────────────────────────
stats = df.groupby("dataset").agg(
    mean_gen_sim=("ref_gen_similarity",  "mean"),
    mean_self_sim=("ref_self_similarity", "mean"),
)
stats["embedding_gap"] = stats["mean_self_sim"] - stats["mean_gen_sim"]

print("=== SUMMARY: Humairawan UrduSpeech (BOOK / WIKI / CONV) ===")
display(stats)

# ── Save CSV ──────────────────────────────────────────────────────────────────
csv_path = "humair_style_evaluation_results.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n📄 Full results saved → {csv_path}")

# ── Zip MUSHRA audio ──────────────────────────────────────────────────────────
zip_path = "mushra_humair_styles.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for folder_name in ["mushra_humair_book", "mushra_humair_wiki", "mushra_humair_conv"]:
        if os.path.exists(folder_name):
            for root, _, files in os.walk(folder_name):
                for file in files:
                    zf.write(os.path.join(root, file))
print(f"📦 MUSHRA audio zipped → {zip_path}")
